# Task 4 · Sentiment Analysis
**Objective:** classify text as *positive*, *negative* or *neutral* using classical ML (Naive Bayes + Logistic Regression + Linear SVM).

**Dataset (self-sourced from Kaggle):** *Twitter Entity Sentiment Analysis* (`twitter_training.csv`, columns: `id, entity, sentiment, text`).
Any CSV with a text column and a sentiment column works - just edit the config cell below.

**Setup:** `pip install pandas scikit-learn nltk matplotlib seaborn wordcloud`

In [ ]:
import re, string, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 1. Load dataset & inspect class distribution

In [ ]:
# ---- CONFIG: edit for your downloaded Kaggle file ----
DATA_PATH   = "twitter_training.csv"   # path to the CSV
HAS_HEADER  = False                    # twitter_training.csv has no header row
COLUMNS     = ["id", "entity", "sentiment", "text"]  # used only if HAS_HEADER is False
TEXT_COL, LABEL_COL = "text", "sentiment"
# ------------------------------------------------------

try:
    df = pd.read_csv(DATA_PATH, header=0 if HAS_HEADER else None, names=None if HAS_HEADER else COLUMNS)
except FileNotFoundError:
    # Tiny built-in sample so the notebook still runs end-to-end. Replace with the real dataset!
    print("Dataset not found - using a tiny demo sample. Download the Kaggle CSV for real results.")
    import itertools
    things = ["phone", "app", "hotel", "movie", "laptop", "game", "restaurant", "update"]
    pos_w = ["amazing", "fantastic", "excellent", "wonderful", "brilliant", "love it", "great quality", "very happy with it"]
    neg_w = ["terrible", "awful", "horrible", "broken", "disappointing", "hate it", "poor quality", "very unhappy with it"]
    neu_w = ["released on monday", "has three modes", "comes in black", "opens at nine", "is version two", "ships next week", "was mentioned today", "is on the list"]
    rows = []
    for lab, ws in [("Positive", pos_w), ("Negative", neg_w), ("Neutral", neu_w)]:
        for t, w in itertools.product(things, ws):
            rows.append((f"The {t} is {w}" if lab != "Neutral" else f"The {t} {w}", lab))
    df = pd.DataFrame(rows, columns=[TEXT_COL, LABEL_COL])
df = df[[TEXT_COL, LABEL_COL]].dropna().drop_duplicates().reset_index(drop=True)
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()

# Task needs 3 classes: drop the 'irrelevant' label if the dataset has one
df = df[df[LABEL_COL].isin(["positive", "negative", "neutral"])].reset_index(drop=True)
print(df.shape)
df.head()

In [ ]:
print(df[LABEL_COL].value_counts())
print()
print((df[LABEL_COL].value_counts(normalize=True)*100).round(1).astype(str) + " %")

## 2. Visualise sentiment distribution

In [ ]:
order = ["positive", "neutral", "negative"]
colors = {"positive": "#4caf50", "neutral": "#9e9e9e", "negative": "#e53935"}
ax = sns.countplot(data=df, x=LABEL_COL, order=order, palette=colors)
for c in ax.containers:
    ax.bar_label(c)
ax.set_title("Sentiment distribution in the dataset")
ax.set_xlabel("Sentiment"); ax.set_ylabel("Count")
plt.show()

## 3. Text preprocessing pipeline
Steps: lowercase → remove URLs/@mentions → remove punctuation & digits → tokenise → remove stopwords → stem.

I use a **stemmer (Porter)** because it needs no extra downloads; a lemmatizer (`WordNetLemmatizer`) is a drop-in alternative if you run `nltk.download('wordnet')`.

In [ ]:
try:
    from nltk.corpus import stopwords
    STOP = set(stopwords.words("english"))
except LookupError:
    try:
        nltk.download("stopwords", quiet=True)
        from nltk.corpus import stopwords
        STOP = set(stopwords.words("english"))
    except Exception:
        STOP = set(ENGLISH_STOP_WORDS)   # offline fallback
# keep negations - they flip sentiment ("not good")
STOP -= {"not", "no", "nor", "never", "n't"}

stemmer = PorterStemmer()
PUNCT_TABLE = str.maketrans("", "", string.punctuation)

def preprocess(text):
    text = str(text).lower()                                # 1. lowercase
    text = re.sub(r"http\S+|www\.\S+|@\w+", " ", text)      #    strip URLs / mentions
    text = text.translate(PUNCT_TABLE)                      # 2. remove punctuation
    text = re.sub(r"\d+", " ", text)                        #    remove digits
    tokens = text.split()                                   # 3. tokenise (whitespace after cleaning)
    tokens = [t for t in tokens if t not in STOP]           # 4. stopword removal
    tokens = [stemmer.stem(t) for t in tokens]              # 5. stemming
    return " ".join(tokens)

print(preprocess("I LOVED this game!!! Check https://x.co @friend - it's not bad, 10/10"))
df["clean"] = df[TEXT_COL].apply(preprocess)
df = df[df["clean"].str.len() > 0].reset_index(drop=True)
df[[TEXT_COL, "clean", LABEL_COL]].head()

## 4. WordCloud per sentiment class

In [ ]:
try:
    from wordcloud import WordCloud
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, lab in zip(axes, order):
        text = " ".join(df.loc[df[LABEL_COL] == lab, "clean"])
        wc = WordCloud(width=600, height=400, background_color="white", colormap="viridis").generate(text)
        ax.imshow(wc, interpolation="bilinear"); ax.axis("off"); ax.set_title(lab.capitalize(), fontsize=14)
    plt.tight_layout(); plt.show()
except ImportError:
    print("Run: pip install wordcloud")

## 5. Feature extraction - TF-IDF
**What it is:** TF-IDF (*Term Frequency - Inverse Document Frequency*) turns each text into a numeric vector so a model can learn from it.

- **TF** - how often a word appears in a document.
- **IDF** - down-weights words that appear in *many* documents (e.g. "game", "get"), because they carry little information.
- **TF x IDF** - a word scores high when it is frequent in *this* text but rare overall, so distinctive sentiment words ("awful", "fantast") stand out.

Compared with plain word counts, TF-IDF reduces the influence of filler words. Here we also add **bigrams** (`ngram_range=(1,2)`) so phrases like "not good" are captured.

In [ ]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2 if len(df) > 1000 else 1)
X_text = df["clean"]
y = df[LABEL_COL]

## 6. Train / test split (80/20)

In [ ]:
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    X_text, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

# Fit TF-IDF on TRAIN only (avoids data leakage), then transform both
X_train = tfidf.fit_transform(X_train_txt)
X_test  = tfidf.transform(X_test_txt)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 7. Train classifiers (Naive Bayes + Logistic Regression + Linear SVM)

In [ ]:
models = {
    "Naive Bayes":         MultinomialNB(alpha=0.5),
    "Logistic Regression": LogisticRegression(max_iter=1000, C=2.0, random_state=RANDOM_STATE),
    "Linear SVM":          LinearSVC(C=1.0, random_state=RANDOM_STATE),
}
preds = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds[name] = model.predict(X_test)
    print(f"trained {name}")

## 8. Evaluation - accuracy, precision, recall, F1, confusion matrix

In [ ]:
rows = []
for name, p in preds.items():
    print("="*60); print(name); print("="*60)
    print(classification_report(y_test, p, labels=order, digits=3))
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, p, average="macro", zero_division=0)
    rows.append({"Model": name, "Accuracy": accuracy_score(y_test, p),
                 "Precision (macro)": prec, "Recall (macro)": rec, "F1 (macro)": f1})
results = pd.DataFrame(rows).set_index("Model").round(4)
results

In [ ]:
fig, axes = plt.subplots(1, len(preds), figsize=(6*len(preds), 5))
for ax, (name, p) in zip(np.atleast_1d(axes), preds.items()):
    cm = confusion_matrix(y_test, p, labels=order)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=order, yticklabels=order, ax=ax, cbar=False)
    ax.set_title(f"{name}\nacc = {accuracy_score(y_test, p):.3f}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()

results.plot(kind="bar", figsize=(9, 4), ylim=(0, 1)); plt.title("Model comparison (macro averages)")
plt.xticks(rotation=0); plt.legend(loc="lower right"); plt.show()

## 9. Error analysis - 5 misclassified examples

In [ ]:
best_name = results["F1 (macro)"].idxmax()
print("Best model:", best_name)

err = pd.DataFrame({"original_text": df.loc[X_test_txt.index, TEXT_COL],
                    "cleaned": X_test_txt, "actual": y_test, "predicted": preds[best_name]})
err = err[err["actual"] != err["predicted"]]
print(f"{len(err)} misclassified of {len(y_test)} ({len(err)/len(y_test):.1%})")
pd.set_option("display.max_colwidth", 140)
err.sample(min(5, len(err)), random_state=RANDOM_STATE)[["original_text", "actual", "predicted"]]

**Why models misclassify (discussion - check against the 5 rows above and edit to match):**

1. **Sarcasm / irony** - "Great, another crash" looks positive on the surface ("great") but is negative. Bag-of-words ignores tone.
2. **Negation & context lost in preprocessing** - stemming/stopword removal and the 1-2 gram window can't capture long-range negation ("I thought it would be good, but it wasn't").
3. **Neutral vs. polar confusion** - neutral text often contains a mildly emotional word, so it gets pulled toward positive/negative; this is usually the largest source of error in the confusion matrix.
4. **Mixed sentiment** - one sentence praising one thing and criticising another has no single correct label.
5. **Noisy text / slang / emojis / typos** - removed or unseen at training time, so TF-IDF has no signal.
6. **Class imbalance & label noise** - crowd-labelled tweets are often inconsistent, putting a ceiling on accuracy.

## 10. Predict on new text

In [ ]:
best_model = models[best_name]
def predict_sentiment(text):
    return best_model.predict(tfidf.transform([preprocess(text)]))[0]

for s in ["I absolutely love this phone!", "This is the worst service I've ever had.", "The meeting is at 3 pm on Friday."]:
    print(f"{s!r:55} -> {predict_sentiment(s)}")

## 11. Conclusion
- **Best model:** see the comparison table & bar chart above (`best_name` is picked automatically by macro-F1). On typical Twitter/Amazon data, **Logistic Regression and Linear SVM** usually edge out **Naive Bayes** because they learn discriminative weights and handle correlated n-gram features better, while Naive Bayes assumes feature independence. Naive Bayes is, however, the fastest to train and a strong baseline on small data.
- **Weak spot:** the *neutral* class and sarcastic/mixed text (see error analysis).
- **Real-world applications:** brand/social-media monitoring, customer-review and support-ticket triage, product feedback dashboards, and public-opinion tracking around campaigns or launches.
- **Next steps:** try a lemmatizer, class weighting, hyper-parameter search (`GridSearchCV`), a TextBlob/VADER rule-based baseline, an LSTM, or a fine-tuned transformer (e.g. BERT) to capture context and sarcasm.